# Supplementary Material: Semi-Analytical Calculations

**Associated manuscript:**  
Ferreira, C., Li, W., Eisenbart, B., & Otto, K.  
*An Analytic Methodology for Evaluating Power Meter Features in Fault Detection and Remaining Useful Life Prediction*  
Manuscript submitted to the *International Journal of Prognostics and Health Management*.

## Purpose

This Jupyter Notebook provides supplementary computational material supporting the analytical evaluations presented in the associated manuscript.

The notebook aims to improve the transparency and reproducibility of the analytical results by providing executable Python implementations of the relevant mathematical expressions, assumptions, and numerical calculations.

Specifically, the calculations support the evaluation of:

1. **Fault sensitivity:** Analytical assessment of power signals and derived features using the local approximation of the Kullback–Leibler divergence.
2. **Sampling requirements:** Evaluation of the sensitivity retained under different signal sampling conditions.
3. **Healthy-state variability:** Quantification of fault-induced feature changes relative to normal operating variability.

These evaluations correspond to Steps 2–4 of the proposed methodology and are applied to the tandem cold mill (TCM) and polymer extrusion case studies. In the case of the TCM, the hydraulic actuator leak is not detailed here, as sufficient information is available in the manuscript. 

## Scope and Usage

The notebook is intended as a supplementary resource to the manuscript rather than a standalone implementation of the complete methodology. The interpretation of the results is presented in the associated publication.

Readers are encouraged to consult the manuscript alongside this notebook for the complete methodological context.

In [2]:
import sys
import numpy as np
import sympy as sp
import pandas as pd

print('Python:', sys.version.split()[0])
print('NumPy:', np.__version__)
print('SymPy:', sp.__version__)
print('pandas:', pd.__version__)

Python: 3.12.4
NumPy: 2.0.2
SymPy: 1.14.0
pandas: 2.3.3


# TCM Case Studies Calculations

This section of the notebook presents the analytical results for the Roll Wear and Eccentricity faults.\
The HGC leak fault is excluded for reasons presented in 4.2.2 of the manuscript. 

## Roll Wear

### Step 1 — IFSA Modelling

Neglecting power losses, lubrication load, hydraulic gap-control power, and energy storage, the roll-wear power model is Eq. (13):
$$
P_{\mathrm{TCM}}(\theta_w)
=
P_{\mathrm{drive}}+P_{\mathrm{hydro}}
=
\frac{2F(\theta_w)\omega(\theta_w)a(\theta_w)}
{D(\theta_w)}
$$

This equation is supported in W. Roberts, "Cold Rolling of Steel", 1978, pp570-574\
doi: 10.1201/9781315139661 

### Step 2 — Roll Wear Fault Sensitivity

Following Eq. (13) of the manuscript, the total power consumption is approximated as:

$$
P_{\mathrm{TCM}}(\theta_w)
=
P_{\mathrm{drive}}+P_{\mathrm{hydro}}
=
\frac{2F(\theta_w)\omega(\theta_w)a(\theta_w)}
{D(\theta_w)}
$$

where:

- $F$ is the applied rolling force,
- $\omega$ denotes the roll surface speed in this formulation,
- $a$ is the effective rolling-force lever arm,
- $D$ is the roll radius,
- $\theta_{w}$ is the roll-wear magnitude.

Roll wear is modelled as a progressive reduction in roll radius:

$$
D(\theta_w)=D_0-\theta_w
$$

where $D_0$ denotes the initial roll diameter.

The power expression with respect to the wear magnitude is smooth, making it possible to differentiate, which gives Eq. (14):

$$
\begin{aligned}
\frac{ P}{ \theta_w}
={}&\frac{2}{D_0-\theta_w}
\left[
\omega a\frac{ F}{\theta_w}
+Fa\frac{\omega}{d\theta_w}
+F\omega\frac{d a}{d\theta_w}
\right]\ 
&+\frac{2F\omega a}{(D_0-\theta_w)^2}
\end{aligned}
$$

Under the stated modelling assumptions, $F$ (controlled), $\omega$ (fixed) and $a$ (neglictible change) are independent of the wear magnitude. Consequently:

$$
\frac{d P}{d \theta_w}
=
\frac{2F\omega a}{(D_0-\theta_w)^2}
$$

Evaluating this expression at the healthy condition ($\theta_{w}=0$):

$$
\boxed{
\left.\frac{d P}{d\theta_w}\right|_{\theta_w=0}
=
\frac{2F\omega a}{D_0^2}
}
$$

### Local KL Divergence

To evaluate the effect of the fault sensitivity on the statistical separability of healthy and degraded power measurements, the local approximation of the KL divergence is employed.

Assuming that the power feature follows a Gaussian distribution with constant variance $\sigma_{P,0}^2$ over the small-fault region, the local KL divergence is approximated by employing Eq. (7) and Eq. (8) of the manuscript:

$$
D_{\mathrm{KL}}(\theta_w)
\approx
\frac{1}{2}
\frac{
\left(
\left.\frac{d P}{d\theta_w}\right|_{\theta_w=0}
\right)^2
}{
\sigma_{P,0}^2
}
\theta_w^2
$$

Substituting the derived power sensitivity:

$$
\boxed{
D_{\mathrm{KL}}(\theta_w)
\approx
\frac{1}{2}
\frac{
\left(\frac{2F\omega a}{D_0^2}\right)^2
}{
\sigma_{P,0}^2
}
\theta_w^2
}
$$

The expression shows that the local KL divergence has a positive quadratic coefficient when the power sensitivity is nonzero and the healthy-state variance is finite and positive.

Therefore, progressive roll wear is theoretically observable through changes in the mean power demand for small, nonzero wear magnitudes.

This establishes local fault sensitivity. Whether the corresponding feature change is sufficiently large to support practical detection is investigated in Step 4, where the expected power shift is compared against healthy-state variability.

In [251]:
# Original parameters
F = 15_000_000.0  # N
v = 8350.0       # mm/s
D_0 = 2*213.0      # mm
a = 6      # mm

# Convert parameters to SI units
v = v / 1000       # m/s
D_0 = D_0 / 1000   # m
a = a / 1000       # m

# Roll-wear sensitivity
dP_dtheta = 2 * F * v * a / D_0**2

print(f"Sensitivity: {dP_dtheta:e} W/m")
print(f"Sensitivity: {dP_dtheta/1000:.2f} W/mm")

Sensitivity: 8.282087e+06 W/m
Sensitivity: 8282.09 W/mm


### Step 3 — Roll Wear Sampling-Rate Evaluation

Step 3 evaluates whether reducing the power sampling frequency affects the fault sensitivity identified in Step 2.

For roll wear, a sampling frequency of $f_{s}=1$ Hz is compared against a reference sampling frequency of $f_{\mathrm{ref}}=4$ Hz, corresponding to sampling intervals of 1 s and 0.25 s, respectively.

Applying Eq. (9) of the manuscript to the results obtained in Eq. (14), the sensitivity-retention ratio is:

$$
R^{\mu,f_s} =
\left|
\frac{
\frac{2F(N_s)va}{(D_0-\theta_w(N_s))^2}
}{
\frac{2F(N_{\mathrm{ref}})va}{(D_0-\theta_w(N_{\mathrm{ref}}))^2}
}
\right|
$$

where $N_s$ and $N_{\mathrm{ref}}$ correspond to the sampling configurations being compared.

Roll wear is assumed to evolve slowly relative to the sampling intervals. Given that one coil takes approximately 250 s to process, the wear magnitude is treated as quasi-static over both sampling intervals. Under this assumption, and with the applied force, roll surface speed and lever arm treated as constant, the local sensitivity is approximately independent of the sampling frequency.

Therefore:

$$
\boxed{R_{D,\mathrm{wear}}^{\mu,1\,\mathrm{Hz}}\approx 1}
$$

This indicates that a 1 Hz sampling rate is theoretically sufficient to preserve the mean-power sensitivity to roll wear under the stated quasi-static conditions.

### Step 4 — Roll Wear Variability Evaluation

Step 3 established that, under the assumed quasi-static degradation conditions, a sampling frequency of 1 Hz is sufficient to retain the mean-power sensitivity to roll wear.

The expected change in mean power consumption can be approximated using a first-order Taylor expansion locally around the healthy condition $\theta_w = 0$ (small fault magnitude):

$$
\Delta \mu_P(\theta_w) =
\mu_P(\theta_w)-\mu_P(0)
\approx
\left.\frac{dP}{d\theta_w}\right|_{\theta_w=0}
\theta_w
$$

Substituting the sensitivity expression derived in Step 2 gives:

$$
\Delta \mu_P(\theta_w)
\approx
\frac{2Fva}{D_0^2}\theta_w
$$

where $F$ is the applied rolling force, $\omega$ is the roll surface speed, $a$ is the effective lever arm, and $D_{0}$ is the initial roll diameter.

Applying Eq. (10) of the manuscript, the expected change is normalised by the healthy-state standard deviation:

$$
S_P(\theta_w)
=
\frac{|\Delta \mu_P(\theta_w)|}{\sigma_{P,0}}
\approx
\frac{1}{\sigma_{P,0}}
\left|\frac{2Fva}{D_0^2}\theta_w\right|
$$

where $\sigma_{P,0}$ is the standard deviation of the power feature under healthy operating conditions.

The following calculation evaluates this quantity for a maximum roll-wear magnitude of 3 mm. A larger standardised shift indicates greater separation between the expected healthy and degraded mean-power values relative to normal operating variability.

In [252]:
import numpy as np

# Physical parameters (SI units)
F = 15_000_000.0  # N
v = 8350.0       # mm/s
D_0 = 2*213.0      # mm
a = 6      # mm

# Convert parameters to SI units
v = v / 1000       # m/s
D_0 = D_0 / 1000   # m
a = a / 1000       # m

# Fault magnitude
theta_w = 0.003     # m, Maximum roll wear

# Healthy-state variability
sigma_P = 11600.0   # W, Healthy power standard deviation

# Step 2: Mean-power sensitivity
dP_dtheta = 2 * F * v * a / D_0**2

# Step 4: Expected mean-power shift
delta_mu_P = dP_dtheta * theta_w

# Standardised feature shift (Eq. 10)
S_P = np.abs(delta_mu_P) / sigma_P

print(f"Power sensitivity: {dP_dtheta:.2e} W/m")
print(f"Expected power shift: {delta_mu_P:.2f} W")
print(f"Standardised feature shift: {S_P:.3f}")

Power sensitivity: 8.28e+06 W/m
Expected power shift: 24846.26 W
Standardised feature shift: 2.142


## Eccentricity

### Step 1 — IFSA Modelling

Neglecting power losses, lubrication load, hydraulic gap-control power, and energy storage, the roll-wear power model is Eq. (13):
$$
P_{\mathrm{TCM}}(\theta_w)
=
P_{\mathrm{drive}}+P_{\mathrm{hydro}}
=
\frac{2F(\theta_w)\omega(\theta_w)a(\theta_w)}
{D(\theta_w)}
$$

This equation is supported in W. Roberts, "Cold Rolling of Steel", 1978, pp570-574\
doi: 10.1201/9781315139661 

### Step 2 — Eccentricity Fault Sensitivity

Following Eq. (13) of the manuscript, the total power consumption is approximated as:

$$
P_{\mathrm{TCM}}(\theta_e)
=
P_{\mathrm{drive}}+P_{\mathrm{hydro}}
=
\frac{2F(\theta_e)\omega(\theta_e)a(\theta_e)}
{D(\theta_e)}
$$

where:

- $F$ is the applied rolling force,
- $\omega$ denotes the roll surface speed in this formulation,
- $a$ is the effective rolling-force lever arm,
- $D$ is the roll radius,
- $\theta_{e}$ is the eccentricity magnitude.

Eccentricity is modelled as a periodic modulation of the roll diameter, described in Eq. (16) of the manuscript:

$$
D(\theta_e)=D_0 + \theta_e sin(\omega t + \phi)
$$

where $D_0$ denotes the initial roll diameter. Injecting this formulation in Eq. (13) of the manuscript, Eq. (13) can be re-written:

$$
\begin{aligned}
P(t, \theta_e) = \frac{1}{d_0} F(t, \theta_e)\omega(t, \theta_e) \frac{2a(t, \theta_e)}{1 + \frac{\theta_e}{d_0} sin(\omega t + \phi)}
\end{aligned}
$$

Because $\frac{\theta_e}{d_0} << 1$, a series expansion around $0$ can be generated:

$$
\begin{aligned}
P(t, \theta_e) = \frac{Q_0}{d_0} - \frac{Q_0}{d_0^2}\theta_e sin(\omega t + \phi) + \frac{Q_0^2}{d_0^3}\theta_e sin^2(\omega t + \phi) - ...
\end{aligned}
$$

with $Q_0 = 2F\omega a$. Each term following the constant is a successive derivative of $P$, and all tend to $0$ when $\theta_e$ tends to 0. Ultimately, this informs that the simple measurement of electrical power is not sensitive to the eccentricity at any order, which motivates the search for other features of the power signal, as stated in the manuscript. Particularly, it is desirable to find a formulation for which a feature of the power signal would be sensitive at the first order. 

### Fourier Transform

For a correctly chosen sampling rate, the sine projection through a discrete Fourier transform is:

$$
\begin{aligned}
Z_e = \frac{2}{N} \sum_{k=1}^{N}{x(t_k)sin(\omega t_k + \phi)}
\end{aligned}
$$

In this case, it is identified through the previous derivation that:

$$
\begin{aligned}
x(t_k) = \frac{Q_0}{d_0} - \frac{Q_0}{d_0^2}\theta_e sin(\omega t_k + \phi) + \varepsilon(t_k)
\end{aligned}
$$

In this expression, $\varepsilon(t_k)$ is random noise. Injecting this expression of $x(t_k)$ into the Fourier Transform formulation, we obtain: 

$$
\begin{aligned}
Z_e = \frac{2}{N} \sum_{k=1}^{N}{[\frac{Q_0}{d_0} - \frac{Q_0}{d_0^2}\theta_e sin(\omega t + \phi)]sin(\omega t_k + \phi)}
\end{aligned}
$$

Substitution yields:

$$
\begin{aligned}
Z_e = \frac{2}{N} \sum_{k=1}^{N}{\frac{Q_0}{d_0} sin(\omega t_k + \phi)} - \frac{2}{N} \sum_{k=1}^{N}{\frac{Q_0}{d_0^2}\theta_e sin^2(\omega t_k + \phi)} + \frac{2}{N} \sum_{k=1}^{N}{\varepsilon(t_k)sin(\omega t_k + \phi)}
\end{aligned}
$$

Over a number of periods:
$$
\begin{aligned}
\frac{2}{N} \sum_{k=1}^{N}{\frac{Q_0}{d_0} sin(\omega t_k + \phi)} = 1
\end{aligned}
$$

Because nothing can be about the value of $\frac{2}{N} \sum_{k=1}^{N}{\varepsilon(t_k)sin(\omega t_k + \phi)}$ due to the random noise, we assume a constant value $\eta$. Therefore, the Fourrier coefficient (here limited to the first harmonic) $Z_e$ has the following expression, the result provided in Eq. (21):
$$
\begin{aligned}
Z_e = -\frac{Q_0}{d_0^2}\theta_e + \eta
\end{aligned}
$$

If we assume the original noise had value $\sigma_0^2$ independent of time $t_k$, then:
$$
\begin{aligned}
Var(\eta) = \frac{4}{N^2}\sigma_0^2 \sum{k=1}^{N} sin^2(\omega t_k+\phi) = \frac{2\sigma_0^2}{N}
\end{aligned}
$$

As such, a complete picture can be given for the values adopted by $Z_e$:
$$
\begin{aligned}
Z_e|\theta_e \sim \mathcal{N}(-\frac{Q_0}{d_0^2}\theta_e, \frac{2\sigma_0^2}{N})
\end{aligned}
$$

### Local KL Divergence

Deriving the mean of $Z_e$ with respect to $\theta_e$ will immediately yield a non-null value, as the evaluation of Eq. (21) shows in the manuscript. Since this sensitivity is of 1st order in mean, it translte to a second order sensitivity in terms of Kullback-Liebler divergence.\
The main conclusion is thus that fault-sensitivity for this feature exists, and thus makes it valid candidate for fault detection and RUL prediction.  

In [253]:
# Original parameters
F = 15_000_000.0  # N
v = 8350.0       # mm/s
D_0 = 2*213.0      # mm
a = 6      # mm

# Convert parameters to SI units
v = v / 1000       # m/s
D_0 = D_0 / 1000   # m
a = a / 1000       # m

Q_0 = 2 * F * v * a

# Roll-wear sensitivity
dZ_dtheta = Q_0 / D_0**2

print(f"Sensitivity: {dZ_dtheta:e} W/m")
print(f"Sensitivity: {dZ_dtheta/1000:.2f} W/mm")

Sensitivity: 8.282087e+06 W/m
Sensitivity: 8282.09 W/mm


### Step 3 — Eccentricity: Sampling-Rate Evaluation

The previous step established:
$$
\begin{aligned}
Z_e = \frac{2}{N} \sum_{k=1}^{N}{\frac{Q_0}{d_0} sin(\omega t_k + \phi)} - \frac{2}{N} \sum_{k=1}^{N}{\frac{Q_0}{d_0^2}\theta_e sin^2(\omega t_k + \phi)} + \frac{2}{N} \sum_{k=1}^{N}{\varepsilon(t_k)sin(\omega t_k + \phi)}
\end{aligned}
$$

Derivation of this formulation of $Z_e$ with respect to $\theta_e$ yields: 
$$
\begin{aligned}
\frac{dZ_e}{d\theta_e} = - \frac{2}{N} \frac{Q_0}{d_0^2}\sum_{k=1}^{N}{sin^2(\omega t_k + \phi)}
\end{aligned}
$$

Let the signal be sampled at frequency $f_{s}$ with $T_s -  \frac{1}{f_s}$ and $t_k = t_0 + (k-1)T_s$.\
Directly $\omega t_k + \phi = \omega t_0 + \phi + (k-1)\omega T_s$.
Define: $\psi = \omega t_0 + \phi$ and $\Delta = \omega T_s = \frac{\omega}{T_s}$.

The sensitivity can be re-written:
$$
\begin{aligned}
\frac{dZ_e(f_s)}{d\theta_e} = - \frac{2}{N} \frac{Q_0}{d_0^2}\sum_{k=1}^{N}{sin^2(\psi + (k-1)\Delta)}
\end{aligned}
$$

Applying Eq. (9) yields Eq. (26) of the manuscript:
$$
\begin{aligned}
R^{Z, f_s} = \frac{|\frac{dZ_e(f_s)}{d\theta_e}|}{|\frac{dZ_e}{d\theta_e}|} = |\frac{- \frac{2}{N} \frac{Q_0}{d_0^2}\sum_{k=1}^{N}{sin^2(\psi + (k-1)\Delta)}}{\frac{dZ_e}{d\theta_e}}|
\end{aligned}
$$
$$
\begin{aligned}
R^{Z, f_s} = |\frac{- \frac{2}{N} \frac{Q_0}{d_0^2}\sum_{k=1}^{N}{sin^2(\psi + (k-1)\Delta)}}{\frac{Q_0}{d_0^2}}| = \frac{2}{N}\sum_{k=1}^{N}{sin^2(\psi + (k-1)\Delta)}
\end{aligned}
$$

If the sampled phase points cover the eccentricity cycle adequately, then $\sum_{k=1}^{N}{sin^2(\psi + (k-1)\Delta)} \approx \frac{N}{2}$ and as such $R^{(Z, f_s)} \approx 1$. However, if $f_s$ becomes too low, the phase points become sparse and the sampled average of $sin^2$ deviates from $\frac{N}{2}$ leading to $R^{(Z, f_s)} < 1$.\
\
Below we demonstrate how we achieve the bounded equation  Eq. (27) in the manuscript.\
Let's define $S_N = \frac{2}{N}\sum_{k=1}^{N}{sin^2(\psi + (k-1)\Delta)}$ and remember that $sin^2(x) = \frac{1-cos(2x)}{2}$.

First, we find that $R^{(Z, f_s)} = \frac{2}{N}S_N$ and that when $S_N = \frac{N}{2}$, then $R^{(Z, f_s)} = 1$.\
Second, $S_N = \frac{N}{2}-\frac{sin(N\Delta)}{2sin(\Delta)}cos(2\psi + (N-1)\Delta)$ provided that $sin(\Delta) \neq 0$ (sampling out of phase).

Ideally, we would want $R^{(Z, f_s)}$ to deviate as little as possible from 1, which can be tracked by calculating $|R^{(Z, f_s)}-1| = |\frac{2 S_N}{N}-1|$.\
From the expression of $S_N$, $|R^{(Z, f_s)}-1| = |\frac{sin(N\Delta)}{2sin(\Delta)}cos(2\psi + (N-1)\Delta)|$.

By definition, $|sin(N\Delta)|\leq 1$ and $|cos(2\psi + (N-1)\Delta)| \leq 1$, therefore a conservative upper bound is Eq. (27) of the manuscript:
$$
\begin{aligned}
|R^{(Z, f_s)}-1| \leq \frac{1}{N|sin(\Delta)|}
\end{aligned}
$$


### Step 4 — Eccentricity Variability Evaluation

Details for Step 3 in the manuscript established that a sampling frequency of 10 Hz is here sufficient to retain the harmonic sensitivity to eccentricity.

The expected change in harmonic magnitude can be approximated using a first-order Taylor expansion locally around the healthy condition $\theta_e = 0$ (small fault magnitude):

$$
\Delta \mu_{Z_e}(\theta_w) =
\mu_{Z_e}(\theta_e)-\mu_{Z_e}(0)
\approx
\left.\frac{dZ_e}{d\theta_e}\right|_{\theta_e=0}
\theta_e
$$

Substituting the sensitivity expression derived in Step 2 gives:

$$
\Delta \mu_{Z_e}(\theta_e)
\approx
-\frac{Q_0}{d_0^2}*\theta_e
$$

Applying Eq. (10) of the manuscript, the expected change is normalised by the healthy-state standard deviation:

$$
S_{Z_e}(\theta_e)
=
\frac{|\Delta \mu_{Z_e}(\theta_e)|}{\sigma_{Z_e,0}}
\approx
\frac{1}{\sigma_{Z_e,0}}
\left|\frac{Q_0}{D_0^2}\theta_e\right|
$$
where $\sigma_{Z_e,0}$ is the standard deviation of the harmonic magnitude feature under healthy operating conditions.

The following calculation evaluates this quantity for a maximum eccentricity magnitude of 1*10e-7 mm, which already produces strong waving in the produced steel sheets. A larger standardised shift indicates greater separation between the expected healthy and degraded mean-power values relative to normal operating variability.

In [254]:
import numpy as np

# Physical parameters (SI units)
F = 15_000_000.0  # N
v = 8350.0       # mm/s
D_0 = 2*213.0      # mm
a = 6      # mm

# Convert parameters to SI units
v = v / 1000       # m/s
D_0 = D_0 / 1000   # m
a = a / 1000       # m

# Fault magnitude
theta_e = 1*10e-7    # m, Maximum roll wear

# Healthy-state variability
sigma_Z = 4.32*10e-1   # W, Healthy power standard deviation

# Step 2: Mean-power sensitivity
dZ_dtheta = 2 * F * v * a / D_0**2

# Step 4: Expected mean-power shift
delta_mu_Z = dZ_dtheta * theta_e

# Standardised feature shift (Eq. 10)
S_Z = np.abs(delta_mu_Z) / sigma_Z

print(f"Power sensitivity: {dZ_dtheta:.2e} W/m")
print(f"Expected power shift: {delta_mu_Z:.2f} W")
print(f"Standardised feature shift: {S_Z:.3f}")

Power sensitivity: 8.28e+06 W/m
Expected power shift: 8.28 W
Standardised feature shift: 1.917


# Extrusion Case Study Calculations

### Step 1 —  Extrusion Clog IFSA Modelling

The IFSA modelling presented in Figure 6 leads to a power equation containing four terms. The screw and its motor contribute two terms, one for the base load and one modelling the advance and exit of material from the barrel with each rotation. Heaters contribute one term and losses exist. It yields Eq. (29) in the manuscript: 
$$
P_{\mathrm{extruder}}(t)
=
\bar{P}_{\mathrm{screw}}(t) + \sum_{m=1}^{M}A_m sin(m\omega_s t + \varphi_m) + P_{\mathrm{heater}}(t)
$$

### Step 2 — Extrusion Fault Sensitivity

Filter clogging during extrusion operation effectively reduces the outlet area.\
The effective area is modelled using a dimensionaless clogging magnitude $\theta_c \in [0,1]$, with $1$ representing a fully clogged outlet. Eq. (30) models its impact on the effective outlet area: 

$$
A_{\mathrm{eff}}
=
A_{\mathrm{0}}(1-\theta_c)
$$

To demonstrate the result of Eq. (31), we start with the approximate flow relation:

$$
Q
=
C_d A_{eff} \sqrt{\frac{2\Delta p}{\rho}}
$$

By replacing $A_{eff}$ with its faulty formulation and isolating $\Delta p$, we obtain:

$$
\Delta p(\theta_c)
=
\frac{\rho Q^3}{2C_d^2A_0^2(1-\theta_c)^2}
= \frac{P_0}{(1-\theta_c)^2}
$$

with $P_0 = \frac{\rho Q^3}{2C_d^2A_0^2}$. It is the result of Eq. (31). Deriving $P_extruder$ with respect to $\theta_c$ would yield a non-null sensitivity, however because there is signal modulation, it can be worth also investigating the sensitivity of the harmonics.\
Another challenge, due to presence of heaters in the system that trigger on and off generating large variability, might disqualify the simple use of the power signal at Step 4.

We know demonstrate the lead up to Eq. (32) to (34) in the manuscript. For calculation comfort, we extract the harmonic amplitude at frequency $m \omega_s$ using the cosine projection:
$$
Z_{m,c}
=
\frac{2}{N}\sum_{k=1}^{N}P_{extruder}(t_k, \theta_c)cos(m \omega_s t_k)
$$

By substituting $P_extruder$ with its formulation in Eq. (29), we obtain:
$$
Z_{m,c}
=
\frac{2}{N}\sum_{k=1}^{N}P_{heater}(t_k)cos(m \omega_s t_k) + \frac{2}{N}\sum_{k=1}^{N}\bar{P}_{screw}(\theta_c)cos(m \omega_s t_k) + \frac{2}{N}\sum_{k=1}^{N}\sum_{r=1}^{M}A_r(\theta_c)cos(r\omega_s t_k + \varphi_r)cos(m\omega_s t_k) + \frac{2}{N}\sum_{k=1}^{N}P_{loss}(t_k)cos(m \omega_s t_k)
$$

For sufficiently uniformly sampled cycles, $\sum_{k=1}^{N}cos(m\omega_s t_k) \approx 0$.

Therefore:
$$
\frac{2}{N}\sum_{k=1}^{N}\bar{P}_{screw}(\theta_c)cos(m \omega_s t_k) \approx 0
$$

For the harmonic sum, terms with $r \neq m$ vanish:
$$
\frac{2}{N}\sum_{k=1}^{N}cos(r\omega_s t_k + \varphi_r)cos(m\omega_s t_k) \approx 0
$$

As such, the retained term in the $m-th$ harmonic, where $\eta_{m,c}$ is a noise term containing heater effects, noise and approximations due to losses. The latter is in practice quite negligible in the frequency domain of the Fourier transform.

$$
Z_{m,c}
=
\frac{2 A_m(\theta_c)}{N} \sum_{k=1}^{N} cos(m \omega_s t_k + \varphi_m)cos(m \omega_s t_k) + \eta_{m,c}
$$

Using $cos(\alpha + \beta) = cos(\alpha)cos(\beta) - sin(\alpha)sin(\beta)$ with $\alpha = m \omega_s t_k$. Hence: 

$$
Z_{m,c}
=
\frac{2 A_m(\theta_c)}{N} cos(\phi_m) \sum_{k=1}^{N}cos^2(m \omega_s t_k) - \frac{2 A_m(\theta_c)}{N} sin(\phi_m) \sum_{k=1}^{N}sin(m \omega_s t_k)cos(m \omega_s t_k) + \eta_{m,c}
$$

Under sufficient sampling:
$$
\frac{1}{N}\sum_{k=1}^{N}cos^2(m \omega_s t_k) \approx \frac{1}{2}
$$
$$
\frac{1}{N}\sum_{k=1}^{N}sin(m \omega_s t_k)cos(m \omega_s t_k) \approx 0
$$

Therefore:

$$
Z_{m,c}(\theta_c)
\approx
A_m(\theta_c) cos(\varphi_m) + \eta_{m,c}
$$

The same demonstration can be done for the $sin$ projection term:
$$
Z_{m,s}(\theta_c)
\approx
-A_m(\theta_c) sin(\varphi_m) + \eta_{m,s}
$$

Ignoring the noise terms, to calculate the amplitude of the harmonic:
$$
Z_{m,c}(\theta_c)^2 + Z_{m,s}(\theta_c)^2
=
A_m(\theta_c)^2 cos^2(\varphi_m) + A_m(\theta_c)^2 sin^2(\varphi_m)
$$

Using $cos^2(\varphi_m) + sin^2(\varphi_m) = 1$, we can devise the amplitude of the harmonic, inclusive of a noise term:
$$
H_{m}(\theta_c)
=
A_m(\theta_c) + \eta_{H_m}
$$

The harmonic amplitude increases with the load, in a first approach a linear model can be devised: 
$$
A_m(\theta_c) = A_{m,0} + \beta_m[\bar(P)_{screw}(\theta_c)-P_0]
$$

Using the formulation for $\bar(P)_{screw}(\theta_c)$ from Eq. (31):
$$
A_m(\theta_c) = A_{m,0} + \beta_m P_0[\frac{1}{(1-\theta_c)}-1]
$$

And therefore: 
$$
H_{m}(\theta_c)
=
A_{m,0} + \beta_m P_0[\frac{1}{(1-\theta_c)}-1] + \eta_{H_m}
$$

And now the derivation to obtain the sensitivity values:
$$
\frac{dH_{m}(\theta_c)}{d\theta_c}
=
\frac{dA_{m}(\theta_c)}{d\theta_c}
=
\frac{2\beta_m P_0}{(1-\theta_c)^3}
$$

Evaluating for $\theta_c = 0$ and injecting in Kullback-Leibler divergence will lead to Eq. (33) and (34).

In [16]:
# Original parameters
rho = 760.0       # kg/m³, LDPE melt density
Q = 0.06          # L/min, Volumetric flow rate
C_d = 0.002      # -, Discharge coefficient for a 20mm screw extruder
A_0 = 2.41        # mm², Initial effective outlet area
beta_m = 1        # -, Harmonic load-coupling coefficient
sigma_H = 5       # W, Healthy harmonic standard deviation
theta_c = 0.10    # -, Clogging magnitude (10%)

# Convert parameters to SI units
Q = Q / (1000 * 60)  # m³/s
A_0 = A_0 * 1e-6     # m²

# Reference hydraulic power
P_0 = rho * Q**3 / (2 * C_d**2 * A_0**2)

# Harmonic amplitude sensitivity at theta_c = 0
dH_dtheta = 2 * beta_m * P_0

# Local KL divergence
D_KL = 0.5 * (dH_dtheta / sigma_H)**2 * theta_c**2

print(f"Reference power: {P_0:.2f} W")
print(f"Sensitivity: {dH_dtheta:.4f} W/unit clogging")

Reference power: 16.36 W
Sensitivity: 32.7129 W/unit clogging


### Step 3 — Extrusion Clogging Sampling-Rate Evaluation

Step 3 evaluates whether the selected power sampling frequency preserves the fault sensitivity of the harmonic features identified in Step 2. It follows the same principle as the demonstration for the roll eccentricity, therefore is not reproduced here.

Although filter clogging develops slowly, the informative power variations occur at the harmonics of the modulated screw load. Consequently, the sampling frequency must be sufficient to capture these periodic variations.

For harmonic order ($m$), the corresponding frequency is:

$$
f_m=mf_r=\frac{m\omega_s}{2\pi}
$$

where ($f_r$) is the screw-rotation frequency and ($\omega_s$) is its angular velocity.

Sampling Sensitivity Retention

Following the same approach as for roll eccentricity, we evaluate the sensitivity of the sampled harmonic feature.

For a sampling frequency ($f_s$), the phase increment between consecutive observations is:

$$
\Delta_m=\frac{2\pi f_m}{f_s}
$$

Assuming a phase-matched sine projection, the sampled feature sensitivity is Eq. (36) of the manuscript:

$$
\frac{2}{N}
\frac{dA_m}{d\theta_c}
\sum_{k=1}^{N}
\sin^2\left(\psi_m+(k-1)\Delta_m\right)
$$

where:
- $N$ is the number of sampled observations.
- $\psi_m$ is the initial harmonic phase.
- $\Delta_m$ is the phase increment between samples.
- $A_m$ is the harmonic amplitude obtained in Step 2.

Normalising the sampled sensitivity by its ideal value gives the sensitivity-retention ratio:

$$
\frac{2}{N}
\sum_{k=1}^{N}
\sin^2\left(\psi_m+(k-1)\Delta_m\right)
$$

This expression has the same form as the sensitivity-retention ratio derived for roll eccentricity.

For sufficiently uniform phase coverage:

$$
\frac{1}{N}\sum_{k=1}^{N}
\sin^2\left(\psi_m+(k-1)\Delta_m\right)
\approx\frac12
$$

Therefore:

$$
\boxed{R_D^{(Z_{m,s},f_s)}\approx1}
$$


**Application to the Polymer Extruder**

During the filter-clogging experiments, power measurements were acquired at the experimental sampling frequency $f_s = 1000 \space Hz$, while the reported screw-motor frequency was approximately $60 \space Hz$.

The following numerical evaluation applies the sensitivity-retention expression to the selected harmonic order and experimental sampling conditions.

Under adequate phase coverage, the retention ratio approaches unity, indicating that the sampled projection preserves the ideal harmonic sensitivity to filter clogging.

This conclusion is conditional on the assumed harmonic model and does not, by itself, establish the absence of aliasing or guarantee equivalent detection performance under different sampling configurations.

In [ ]:
import numpy as np

# Original parameters
f_r = 60.0       # Hz, Mechanical rotation frequency, corresponding to 120 rpm
m = 2            # Harmonic order
f_s = 1000.0     # Hz, Power sampling frequency
psi_m = 0.3      # rad, Initial harmonic phase
T = 1.0          # s, Observation duration

# Harmonic frequency
f_m = m * f_r

# Number of samples and phase increment
N = int(f_s * T)
delta_m = 2 * np.pi * f_m / f_s

# Sampling phase points
k = np.arange(N)
phase = psi_m + k * delta_m

# Sensitivity-retention ratio
R_D = (2 / N) * np.sum(np.cos(phase)**2)

print(f"Harmonic frequency: {f_m:.2f} Hz")
print(f"Sampling frequency: {f_s:.2f} Hz")
print(f"Number of samples: {N}")
print(f"Sensitivity retention: {R_D:.4f}")

Harmonic frequency: 120.00 Hz
Sampling frequency: 1000.00 Hz
Number of samples: 1000
Sensitivity retention: 1.0000


### Step 4 — Extrusion Clogging: Variability Evaluation

Details for Step 3 in the manuscript established that the used sampling frequency of 100 Hz is here sufficient to retain the harmonic sensitivity to clogging.

The expected change in harmonic magnitude can be approximated using a first-order Taylor expansion locally around the healthy condition $\theta_c = 0$ (small fault magnitude):

$$
\Delta \mu_{H_m}(\theta_c) =
\mu_{H_m}(\theta_c)-\mu_{H_m}(0)
\approx
\left.\frac{dH_m}{d\theta_c}\right|_{\theta_c=0}
\theta_c
$$

Substituting the sensitivity expression derived in Step 2 gives:

$$
\Delta \mu_{H_m}(\theta_c)
\approx
\frac{2\beta_m P_0}{(1-\theta_c)^3}*\theta_c
$$

Applying Eq. (10) of the manuscript, the expected change is normalised by the healthy-state standard deviation:

$$
S_{H_m}(\theta_c)
=
\frac{|\Delta \mu_{H_m}(\theta_c)|}{\sigma_{H_m,0}}
\approx
\frac{1}{\sigma_{H_m,0}}
\left|\frac{2\beta_m P_0}{(1-\theta_c)^3}\right|*\theta_c
$$
where $\sigma_{H_m,0}$ is the standard deviation of the harmonic magnitude feature under healthy operating conditions.

The following calculation evaluates this quantity for a maximum eccentricity magnitude of 1*10e-7 mm, which already produces strong waving in the produced steel sheets. A larger standardised shift indicates greater separation between the expected healthy and degraded mean-power values relative to normal operating variability.

In [22]:
# Original parameters
rho = 760.0       # kg/m³, LDPE melt density
Q = 0.06          # L/min, Volumetric flow rate
C_d = 0.002       # -, Discharge coefficient
A_0 = 2.41        # mm², Initial effective outlet area
beta_m = 1.0      # -, Harmonic load-coupling coefficient

# Fault magnitude and healthy-state variability
theta_c = 1       # -, Clogging magnitude (max clogging at 100%)
sigma_H = 5.0     # W, Healthy harmonic standard deviation

# Convert parameters to SI units
Q = Q / (1000 * 60)  # m³/s
A_0 = A_0 * 1e-6     # m²

# Reference hydraulic power
P_0 = rho * Q**3 / (2 * C_d**2 * A_0**2)

# Step 2: Harmonic amplitude sensitivity at healthy state
dH_dtheta = 2 * beta_m * P_0

# Step 4: Expected harmonic amplitude shift (first-order)
delta_H = dH_dtheta * theta_c

# Standardised feature shift (Eq. 10)
S_H = abs(delta_H) / sigma_H

print(f"Reference power: {P_0:.2f} W")
print(f"Harmonic sensitivity: {dH_dtheta:.2f} W")
print(f"Expected harmonic shift: {delta_H:.2f} W")
print(f"Standardised feature shift: {S_H:.3f}")

Reference power: 16.36 W
Harmonic sensitivity: 32.71 W
Expected harmonic shift: 32.71 W
Standardised feature shift: 6.543
